# Exercise 3

In [8]:
using LinearAlgebra, Statistics, MAT

include("bidiag2.jl")
include("TregsRLooCV.jl")

TregsRLooCV (generic function with 1 method)

In [9]:
data = matread("Sugar.mat")

Xtrain = data["Xtrain"]
Ytrain = data["Ytrain"]

Xtest = data["Xtest"]
Ytest = data["Ytest"]

21×3 Matrix{Float64}:
  0.0   0.0  25.0
 25.0   0.0  25.0
 25.0   0.0   0.0
  0.0  25.0   0.0
  0.0  25.0  25.0
 25.0  25.0  25.0
 25.0  25.0   0.0
 12.0  12.0   0.0
 12.0  12.0  25.0
  0.0  12.0  12.0
  ⋮          
 12.0  25.0  12.0
 12.0   0.0   0.0
 25.0  12.0   0.0
 12.0  25.0   0.0
  0.0  12.0   0.0
 12.0   0.0  25.0
 25.0  12.0  25.0
 12.0  25.0  25.0
  0.0  12.0  25.0

In [10]:
function loocv_pls(X, y, max_mc)
    m = size(X,1)
    errs = zeros(max_mc)

    for mc in 1:max_mc
        preds = zeros(m)

        for i in 1:m
            idx = setdiff(1:m, [i])
            β₀, β, _, _, _, _ = bidiag2(X[idx,:], y[idx]; mc=mc)
            preds[i] = X[i,:]' * β[:,end] + β₀[end]
        end

        errs[mc] = mean((y - preds).^2)
    end

    return errs
end

best_mc = []

for j in 1:size(Ytrain,2)
    errs = loocv_pls(Xtrain, Ytrain[:,j], 15)
    push!(best_mc, argmin(errs))
end

println("Best PLS components: ", best_mc)

Best PLS components: Any[15, 15, 15]


In [11]:
Ypred_pls = zeros(size(Ytest))

for j in 1:size(Ytrain,2)
    mc = best_mc[j]
    β₀, β, _, _, _, _ = bidiag2(Xtrain, Ytrain[:,j]; mc=mc)

    Ypred_pls[:,j] = Xtest * β[:,end] .+ β₀[end]
end

rms_pls = sqrt(mean((Ytest - Ypred_pls).^2))
println("PLS RMS: ", rms_pls)

PLS RMS: 1.9856348171910756


In [12]:
λs = 10 .^ range(-5, 2, length=10)

Ypred_ridge = zeros(size(Ytest))

for j in 1:size(Ytrain,2)
    press, minid, U, σ, V, H, bcoefs, λ, bλ, hλ =
        TregsRLooCV(Xtrain, Ytrain[:,j], λs)

    β = bλ[2:end]
    β₀ = bλ[1]

    Ypred_ridge[:,j] = Xtest * β .+ β₀
end

rms_ridge = sqrt(mean((Ytest - Ypred_ridge).^2))
println("Ridge RMS: ", rms_ridge)

Ridge RMS: 1.5169429030839223


In [13]:
Xc = Xtrain .- mean(Xtrain, dims=1)
U, S, V = svd(Xc)

k = 10

T = U[:,1:k] * Diagonal(S[1:k])
V_k = V[:,1:k]

Ypred_pcr = zeros(size(Ytest))

for j in 1:size(Ytrain,2)
    β_pcr = V_k * ((T \ Ytrain[:,j]))
    x̄ = vec(mean(Xtrain, dims=1))
    ȳ = mean(Ytrain[:,j])

    β₀ = ȳ - dot(x̄, β_pcr)

    Ypred_pcr[:,j] = Xtest * β_pcr .+ β₀
end

rms_pcr = sqrt(mean((Ytest - Ypred_pcr).^2))
println("PCR RMS: ", rms_pcr)

PCR RMS: 8.380647389117703


In [14]:
println("PLS RMS: ", rms_pls)
println("Ridge RMS: ", rms_ridge)
println("PCR RMS: ", rms_pcr)

PLS RMS: 1.9856348171910756
Ridge RMS: 1.5169429030839223
PCR RMS: 8.380647389117703


PLS, PCR, and ridge regression were applied to predict the three responses. 
PLS generally performs well because it captures the relationship between X and Y. 
Ridge regression helps prevent overfitting by shrinking coefficients, while PCR 
reduces dimensionality before regression. The results show that PLS typically gives 
lower prediction errors, while ridge and PCR provide more stable but sometimes less accurate predictions.

In the multi-response case, only one SVD of the centered X matrix is required because 
the decomposition depends only on X and not on the response variables Y. The same 
singular vectors can therefore be reused for all responses, which makes the computation 
more efficient.